<a href="https://colab.research.google.com/github/tffnnnnnn/Machine_Learning_MASnI3/blob/main/SCIGEN_00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 0: Welcome & Setup

<img src="https://github.com/RyotaroOKabe/APS_demo_SCIGEN/blob/main/assets/scigen_logo.png?raw=1" width="150" align="right">

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

This notebook gets your Colab environment ready for the rest of the tutorial.
We will:
1. Verify GPU access
2. Clone the SCIGEN repository
3. Install all dependencies
4. Download pretrained model weights
5. Verify everything works

> **Run this notebook first**, then move on to the other notebooks in order.

# **日常使用區**

In [ ]:
import os
from pathlib import Path

# ==========================================
# SCIGEN Project Configuration
# ==========================================

PROJECT_DIR = Path("/content/APS_demo_SCIGEN")

os.environ["PROJECT_ROOT"] = str(PROJECT_DIR)
os.environ["HYDRA_JOBS"] = str(PROJECT_DIR / "hydra_jobs")
os.environ["WANDB_DIR"] = str(PROJECT_DIR / "wandb")

# Check project directory
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"SCIGEN project not found: {PROJECT_DIR}"
    )

# Create output directories
Path(os.environ["HYDRA_JOBS"]).mkdir(
    parents=True, exist_ok=True
)
Path(os.environ["WANDB_DIR"]).mkdir(
    parents=True, exist_ok=True
)

print("SCIGEN project configuration completed.")
print("PROJECT_DIR:", PROJECT_DIR)

# **環境修復區**

---
## 0.1 Check GPU availability

Go to **Runtime > Change runtime type** and select **T4 GPU** before running this cell.

In [2]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'GPU available: {gpu_name}')
else:
    print('WARNING: No GPU detected!')
    print('Go to Runtime > Change runtime type > T4 GPU, then re-run this cell.')
    print('The tutorial will still work on CPU, but generation will be much slower.')

GPU available: Tesla T4


---
## 0.2 Clone the SCIGEN repository

We download the full SCIGEN codebase from GitHub. This repository contains:
- The **diffusion model** code (`scigen/`) used for crystal structure generation
- **Generation scripts** (`script/`) for running inference with pretrained models
- **Configuration files** (`conf/`) for model hyperparameters (Hydra-based)
- **Dataset files** (`data/`) including the MP-20 training/test splits
- **Colab notebooks** (`notebooks/`) — including this one

Cloning takes ~10 seconds. If the repository is already present, this step is skipped.

In [3]:
import os, shutil

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

os.chdir('/content')

if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    !git clone {REPO_URL} {PROJECT_DIR}

assert os.path.isdir(os.path.join(PROJECT_DIR, 'scigen')), \
    f'Clone failed — scigen/ not found in {PROJECT_DIR}'

print(f'Repository ready: {PROJECT_DIR}')

Repository ready: /content/APS_demo_SCIGEN


---
## 0.3 Install dependencies/解決安裝環境問題

This installs PyTorch Geometric extensions (matching the Colab PyTorch/CUDA versions)
and the other libraries SCIGEN needs.

> **This takes ~2–5 minutes.** You can read ahead while it runs.

In [4]:
import torch, os

# Detect PyTorch and CUDA versions for compatible wheel installation
_ver_parts = torch.__version__.split('+')[0].split('.')
TORCH_VER = f'{_ver_parts[0]}.{_ver_parts[1]}.0'
if torch.version.cuda:
    CUDA_VER = 'cu' + torch.version.cuda.replace('.', '')
else:
    CUDA_VER = 'cpu'
PYG_WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'
os.environ['PYG_WHEEL_URL'] = PYG_WHEEL_URL
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda or "N/A (CPU)"}')
print(f'PyG wheel index: {PYG_WHEEL_URL}')

PyTorch 2.14.1+cu130, CUDA 13.0
PyG wheel index: https://data.pyg.org/whl/torch-2.14.0+cu130.html


In [5]:
import sys, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
CUDA: 13.0


In [6]:
print(PYG_WHEEL_URL)

https://data.pyg.org/whl/torch-2.14.0+cu130.html


In [2]:
import os
import torch
from pathlib import Path

PROJECT_DIR = "/content/APS_demo_SCIGEN"

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("PYG_WHEEL_URL:", os.environ.get("PYG_WHEEL_URL"))
print("Requirements exists:",
      Path(PROJECT_DIR, "requirements-colab.txt").is_file())

PyTorch: 2.14.1+cu130
CUDA: 13.0
PYG_WHEEL_URL: None
Requirements exists: True


In [3]:
%%time

import sys
import subprocess

PROJECT_DIR = "/content/APS_demo_SCIGEN"

# Install PyG core
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torch_geometric"],
    check=True
)

# Install tutorial dependencies
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "-r", f"{PROJECT_DIR}/requirements-colab.txt"],
    check=True
)

print("Installation completed successfully.")

Installation completed successfully.
CPU times: user 1.63 ms, sys: 39 µs, total: 1.67 ms
Wall time: 3.94 s


In [4]:
import torch
import torch_geometric
import importlib.util

print("=== Environment Check ===")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("PyG:", torch_geometric.__version__)
print("GPU available:", torch.cuda.is_available())

print("\n=== PyG Extensions ===")

for pkg in ["torch_scatter", "torch_sparse",
            "torch_cluster", "pyg_lib"]:
    print(pkg, importlib.util.find_spec(pkg) is not None)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


=== Environment Check ===
PyTorch: 2.14.1+cu130
CUDA: 13.0
PyG: 2.8.1
GPU available: True

=== PyG Extensions ===
torch_scatter False
torch_sparse False
torch_cluster False
pyg_lib False


In [5]:
from pathlib import Path

req = Path("/content/APS_demo_SCIGEN/requirements-colab.txt")

for line in req.read_text().splitlines():
    if any(name in line.lower() for name in
           ["torch", "scatter", "sparse", "cluster", "pyg"]):
        print(line)

# PyTorch + CUDA are provided by the Colab runtime.
# torch-scatter, torch-sparse, torch-cluster must be installed separately
# from the PyG wheel index matching your torch+CUDA version — see setup cell.
torch-geometric>=2.3.0


In [6]:
import requests
from bs4 import BeautifulSoup

url = "https://data.pyg.org/whl/torch-2.11.0+cu130.html"

response = requests.get(url, timeout=20)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
wheels = [a.get_text() for a in soup.find_all("a")]

for package in ["torch_scatter", "torch_sparse", "torch_cluster"]:
    matches = [
        wheel for wheel in wheels
        if package in wheel
        and "cp313" in wheel
        and "linux_x86_64" in wheel
    ]

    print(f"\n{package}:")
    if matches:
        for wheel in matches:
            print("✓", wheel)
    else:
        print("✗ No compatible wheel found")


torch_scatter:
✓ torch_scatter-2.1.2+pt211cu130-cp313-cp313-linux_x86_64.whl

torch_sparse:
✓ torch_sparse-0.6.18+pt211cu130-cp313-cp313-linux_x86_64.whl

torch_cluster:
✓ torch_cluster-1.6.3+pt211cu130-cp313-cp313-linux_x86_64.whl


In [7]:
%pip install --no-index \
    torch-scatter torch-sparse torch-cluster \
    -f https://data.pyg.org/whl/torch-2.11.0+cu130.html

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 9.2 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 113.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 95.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torch-cluster]


In [8]:
import torch
import torch_geometric
import torch_scatter
import torch_sparse
import torch_cluster

print("=== SCIGEN Environment Check ===")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("PyG:", torch_geometric.__version__)
print("GPU available:", torch.cuda.is_available())

# Test torch_scatter
from torch_scatter import scatter_add

device = "cuda" if torch.cuda.is_available() else "cpu"

src = torch.tensor([1., 2., 3., 4.], device=device)
index = torch.tensor([0, 0, 1, 1], device=device)

result = scatter_add(src, index, dim=0)

print("\n=== torch_scatter GPU Test ===")
print("Device:", result.device)
print("Result:", result)

assert torch.allclose(
    result.cpu(), torch.tensor([3., 7.])
)

print("\nAll PyG extension imports and scatter test passed!")

OSError: Could not load this library: /usr/local/lib/python3.13/dist-packages/torch_scatter/_scatter_cuda.so

解決torch_scatter 的 CUDA 擴充模組與目前 Colab 載入的 PyTorch 二進位版本不相容。

In [9]:
import torch
import sys
import subprocess

print("=== Runtime Environment ===")
print("Loaded PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Python:", sys.version)

print("\n=== Installed PyTorch ===")

subprocess.run([
    sys.executable, "-m", "pip",
    "show", "torch", "torch-scatter"
])

=== Runtime Environment ===
Loaded PyTorch: 2.14.1+cu130
CUDA: 13.0
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

=== Installed PyTorch ===


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'show', 'torch', 'torch-scatter'], returncode=0)

In [10]:
%pip install --index-url https://download.pytorch.org/whl/cu130 \
    "torch==2.11.0+cu130" \
    "torchvision==0.26.0+cu130"

Looking in indexes: https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 144.3 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 256.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 67.1 MB/s  0:00:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 MB 321.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.1/531.1 MB 20.3 MB/s  0:00:14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 149.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 26.1 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 68.3 MB/s  0:00:02
  Attempting uninstall: nvidia-cusparselt-cu13
    Found existing installation: nvidia-cusparselt-cu13 0.8.1
    Uninstalling nvidia-cusparselt-cu13-0.8.1:
      Successfully uninstalled nvidia-cusparselt-cu13-0.8.1
  Attempting uninstall: cuda-toolkit
    Found existing installation: cuda-toolkit 13.0.3.0
   

驗證是否安裝成功

In [1]:
import torch
import torch_scatter
import torch_sparse
import torch_cluster

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.is_available())

print("torch_scatter:", torch_scatter.__version__)
print("torch_sparse:", torch_sparse.__version__)
print("torch_cluster:", torch_cluster.__version__)

print("\nEnvironment verification successful!")

PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: True
torch_scatter: 2.1.2+pt211cu130
torch_sparse: 0.6.18+pt211cu130
torch_cluster: 1.6.3+pt211cu130

Environment verification successful!


確認相依套件沒有衝突

In [2]:
!pip check

ipython 7.34.0 requires jedi, which is not installed.
google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.34.2.


修復相依性問題

In [3]:
%pip install "jedi" "requests==2.32.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 136.5 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jedi]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymatgen-core 2026.10.31 requires requests>=2.32.5, but you have requests 2.32.4 which is incompatible.


SCIGEN 研究需要使用 pymatgen 讀取和處理 MASnI₃ 晶體結構，我們應先確認目前版本是否真的影響功能。

In [4]:
import torch
import requests
import pymatgen.core
from pymatgen.core import Lattice, Structure
from torch_scatter import scatter_add

print("=== Package Verification ===")
print("PyTorch:", torch.__version__)
print("Requests:", requests.__version__)
print("GPU:", torch.cuda.is_available())

# Test pymatgen
structure = Structure(
    Lattice.cubic(5.64),
    ["Na", "Cl"],
    [[0, 0, 0], [0.5, 0.5, 0.5]]
)

print("\n=== pymatgen Test ===")
print("Formula:", structure.formula)
print("Atoms:", len(structure))

# Test torch_scatter on GPU
src = torch.tensor([1., 2., 3., 4.], device="cuda")
index = torch.tensor([0, 0, 1, 1], device="cuda")

result = scatter_add(src, index, dim=0)

print("\n=== torch_scatter Test ===")
print("Result:", result)

assert torch.allclose(result.cpu(), torch.tensor([3., 7.]))

print("\nAll functional tests passed!")

=== Package Verification ===
PyTorch: 2.11.0+cu130
Requests: 2.32.4
GPU: True

=== pymatgen Test ===
Formula: Na1 Cl1
Atoms: 2

=== torch_scatter Test ===
Result: tensor([3., 7.], device='cuda:0')

All functional tests passed!


---
## 0.4 Configure environment variables

SCIGEN requires several environment variables to be set **before** importing its modules
(the import chain calls `os.chdir(PROJECT_ROOT)` at load time):

| Variable | Purpose |
|----------|---------|
| `PROJECT_ROOT` | Root directory of the SCIGEN codebase — all relative paths resolve from here |
| `HYDRA_JOBS` | Where Hydra stores configuration outputs (needed for model loading) |
| `WANDB_DIR` | Logging directory (required by the training code, even during inference) |

We also add the project and `script/` directories to `sys.path` so that
`import scigen` and `from gen_utils import ...` work correctly in the notebook.

In [5]:
import os, sys

PROJECT_DIR = '/content/APS_demo_SCIGEN'

os.environ['PROJECT_ROOT'] = PROJECT_DIR
os.environ['HYDRA_JOBS'] = PROJECT_DIR
os.environ['WANDB_DIR'] = os.path.join(PROJECT_DIR, 'wandb')
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'script'))

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print('Environment configured.')

Working directory: /content/APS_demo_SCIGEN
Environment configured.


---
## 0.5 Download pretrained model weights

Downloads the SCIGEN model checkpoint (~250 MB) from Figshare.
This is needed for Notebook 04 (generation).

In [6]:
from pathlib import Path

# SCIGEN 專案目錄
PROJECT_DIR = Path("/content/APS_demo_SCIGEN")

# 模型儲存位置
MODEL_DIR = PROJECT_DIR / "models"
MODEL_PATH = MODEL_DIR / "mp_20"

# 確認專案是否存在
print("Project exists:", PROJECT_DIR.is_dir())
print("Model directory:", MODEL_PATH)

Project exists: True
Model directory: /content/APS_demo_SCIGEN/models/mp_20


In [7]:
# takes long (~2-5 min) — downloads ~250 MB model weights from Figshare

import json, zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve

MODEL_DIR = Path(PROJECT_DIR) / 'models'
MODEL_PATH = MODEL_DIR / 'mp_20'
FIGSHARE_API_URL = 'https://api.figshare.com/v2/articles/27778134'

need_download = not MODEL_PATH.exists() or not list(MODEL_PATH.glob('*.ckpt'))

if need_download:
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    print('Querying Figshare API...')
    article = json.loads(urlopen(FIGSHARE_API_URL).read().decode())
    files = article.get('files', [])
    print(f'Found {len(files)} file(s)')

    for f in files:
        dest = MODEL_PATH / f['name']
        print(f'Downloading {f["name"]}...')
        urlretrieve(f['download_url'], str(dest))
        print(f'  Saved ({dest.stat().st_size / 1e6:.1f} MB)')
        if dest.suffix == '.zip':
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_PATH)
            dest.unlink()
else:
    print('Model already downloaded.')

# Verify
ckpts = list(MODEL_PATH.glob('*.ckpt'))
assert ckpts, f'No .ckpt files in {MODEL_PATH}. Download may have failed.'
print(f'Model ready: {[c.name for c in ckpts]}')

Querying Figshare API...
Found 4 file(s)
  Saved (148.3 MB)
  Saved (0.0 MB)
  Saved (0.0 MB)
  Saved (0.0 MB)
Model ready: ['epoch=819-step=86919.ckpt']


> **Troubleshooting:** If the download fails, try re-running the cell. If it fails repeatedly,
> you can manually download from [Figshare](https://doi.org/10.6084/m9.figshare.27778134)
> and upload the files to `models/mp_20/` in the Colab file browser.

import torch
import torch_geometric
import hydra
from importlib.metadata import version

checks = {
    'torch': torch.__version__,
    'torch_geometric': torch_geometric.__version__,
    'pymatgen': version('pymatgen'),
    'hydra': hydra.__version__,
}

# Optional packages
for pkg in ['chgnet', 'mp_api']:
    try:
        checks[pkg] = version(pkg)
    except Exception:
        checks[pkg] = '(not installed)'

for pkg, ver in checks.items():
    print(f'  {pkg:<20s} {ver}')

print(f'\n  CUDA available:      {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU:                 {torch.cuda.get_device_name(0)}')
print()
print('Setup complete! You are ready for the tutorial.')

In [8]:
import torch
import torch_geometric
import hydra
from importlib.metadata import version

print(f'torch:           {torch.__version__}')
print(f'torch_geometric: {torch_geometric.__version__}')
print(f'pymatgen:        {version("pymatgen")}')
print(f'hydra:           {hydra.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
print()
print('Setup complete! You are ready for the tutorial.')

torch:           2.11.0+cu130
torch_geometric: 2.8.1
pymatgen:        2026.10.24
hydra:           1.3.8
CUDA available:  True

Setup complete! You are ready for the tutorial.


---
## References

- **SCIGEN:** Okabe et al., "Structural constraint integration in a generative model for the discovery of quantum materials," *Nature Materials* (2025). [DOI](https://doi.org/10.1038/s41563-025-02355-y) | [GitHub](https://github.com/RyotaroOKabe/SCIGEN)
- **PyTorch:** Paszke et al., "PyTorch: An Imperative Style, High-Performance Deep Learning Library," *NeurIPS* (2019). [pytorch.org](https://pytorch.org)
- **PyTorch Geometric:** Fey & Lenssen, "Fast Graph Representation Learning with PyTorch Geometric," *ICLR Workshop* (2019). [GitHub](https://github.com/pyg-team/pytorch_geometric)
- **Hydra:** Yadan, "Hydra — A framework for elegantly configuring complex applications," (2019). [GitHub](https://github.com/facebookresearch/hydra)

---
## What's next?

Proceed to **Notebook 01: Crystal Structures** to learn how to work with crystal structures in Python.